# 8-3절 연습 문제 풀이

이 노트북은 8-3절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch08/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
DATA_ROOT = '../../download'

def cifar_loaders(batch_size=64):
    tf = transforms.Compose([transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))])
    full = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=tf)
    test = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=tf)
    g = torch.Generator().manual_seed(SEED)
    n_val = int(len(full) * 0.2)
    tr, va = random_split(full, [len(full) - n_val, n_val], generator=g)
    return (DataLoader(tr, batch_size=batch_size, shuffle=True),
            DataLoader(va, batch_size=batch_size), DataLoader(test, batch_size=batch_size))

def fit(model, epochs=10, lr=1e-3):
    tr, va, te = cifar_loaders()
    model = model.to(device); crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for e in range(1, epochs + 1):
        model.train()
        for x, y in tr:
            loss = crit(model(x.to(device)), y.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); c = n = 0
    with torch.no_grad():
        for x, y in te:
            c += (model(x.to(device)).argmax(1).cpu() == y).sum().item(); n += len(y)
    print(f'  평가 정확도 {c / n * 100:.2f}%')
    return c / n * 100

class ResidualBlock(nn.Module):
    def __init__(self, fan_in, fan_out, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(fan_in, fan_out, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(fan_out)
        self.conv2 = nn.Conv2d(fan_out, fan_out, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(fan_out)
        self.relu = nn.ReLU()
        self.shortcut = nn.Sequential()
        if stride != 1 or fan_in != fan_out:
            self.shortcut = nn.Sequential(
                nn.Conv2d(fan_in, fan_out, 1, stride), nn.BatchNorm2d(fan_out))
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.shortcut(x))

def make_resnet(block, channels=(32, 64, 128)):
    layers, fan_in = [nn.Conv2d(3, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU()], 32
    for ch in channels:
        layers += [block(fan_in, ch), nn.MaxPool2d(2)]; fan_in = ch
    layers += [nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(fan_in, 10)]
    return nn.Sequential(*layers)

## 연습 8-11

ResidualBlock 클래스에 포함된 합성곱 계층은 스트라이드 인자(stride)를 생략해 기본값(1)을 사용한다. 기본값 대신 생성자가 전달받은 값을 사용하도록 ResidualBlock 클래스를 수정해 보자. 수정 후, 스트라이드 인자를 2로 바뀌면 특징 지도의 크기가 변하는지 확인해 보자.

힌트: 주 경로의 크기에 따라 지름길 경로의 합성곱 계층도 수정이 필요하다.

In [ ]:
x = torch.randn(1, 32, 32, 32)
for stride in (1, 2):
    block = ResidualBlock(32, 64, stride=stride)
    print(f'stride={stride}: 입력 {tuple(x.shape)} -> 출력 {tuple(block(x).shape)}')

주 경로의 첫 합성곱에 `stride`를 적용하면 특징 지도가 절반으로 줄어든다. 이때 **지름길 경로도 같은 크기로 맞춰야** 덧셈이 가능하므로, 1×1 합성곱에 같은 stride를 적용해 크기와 채널을 함께 맞춘다. 위 `shortcut`이 그 처리를 담당한다.

## 연습 8-12

MiniResNet 클래스의 AdaptiveAvgPool2d((1, 1))의 인자를 (2, 2)로 바꾸면 마지막 nn.Linear 계층의 인자는 어떻게 바뀔까?

In [ ]:
for out_size in ((1, 1), (2, 2)):
    pool = nn.AdaptiveAvgPool2d(out_size)
    x = torch.randn(1, 128, 8, 8)
    flat = nn.Flatten()(pool(x))
    print(f'AdaptiveAvgPool2d({out_size}): {tuple(pool(x).shape)} -> '
          f'평탄화 {tuple(flat.shape)} -> nn.Linear({flat.shape[1]}, 10)')

`(1, 1)`이면 채널마다 값 하나만 남아 선형 계층의 입력 크기가 **채널 수**와 같다. `(2, 2)`로 바꾸면 채널마다 4개가 남으므로 입력 크기가 **채널 수 × 4**가 된다. 즉 `nn.Linear(128, 10)`이 `nn.Linear(512, 10)`으로 바뀐다.

## 연습 8-13

MiniResNet 모델은 합성곱 계층 두 개마다 지름길을 하나씩 둔 구조다. 합성곱 계층 하나마다 지름길을 하나씩 두도록 클래스를 수정한 후 결과를 확인해 보자.

In [ ]:
class SingleConvResidual(nn.Module):
    """합성곱 하나마다 지름길을 두는 잔차 블록"""
    def __init__(self, fan_in, fan_out, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(fan_in, fan_out, 3, stride, 1)
        self.bn = nn.BatchNorm2d(fan_out)
        self.relu = nn.ReLU()
        self.shortcut = nn.Sequential()
        if stride != 1 or fan_in != fan_out:
            self.shortcut = nn.Sequential(nn.Conv2d(fan_in, fan_out, 1, stride),
                                          nn.BatchNorm2d(fan_out))
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)) + self.shortcut(x))

def make_resnet(block, channels=(32, 64, 128)):
    layers, fan_in = [nn.Conv2d(3, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU()], 32
    for ch in channels:
        layers += [block(fan_in, ch), nn.MaxPool2d(2)]; fan_in = ch
    layers += [nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(fan_in, 10)]
    return nn.Sequential(*layers)

for name, blk in [('합성곱 2개마다 지름길', ResidualBlock),
                  ('합성곱 1개마다 지름길', SingleConvResidual)]:
    torch.manual_seed(SEED)
    model = make_resnet(blk)
    print(f'[{name}] 파라미터 {sum(p.numel() for p in model.parameters()):,}개')
    fit(model, epochs=10)

실행 결과를 보면 차이가 뚜렷하다.

| 구성 | 파라미터 | 평가 정확도 |
|---|---|---|
| 합성곱 2개마다 지름길 (원래 방식) | 309,322개 | **79.39%** |
| 합성곱 1개마다 지름길 | 115,114개 | **63.28%** |

파라미터가 1/3로 줄면서 정확도도 **16%p 넘게 떨어졌다**. 다만 파라미터 수 자체가 크게 달라 공정한 비교는 아니므로, 두 가지를 함께 봐야 한다.

**구조적인 이유**: 잔차 블록은 `출력 = 입력 + F(입력)` 형태로, 블록이 학습하는 것은 '입력에 더할 변화량 F'다. 합성곱 하나에 지름길을 두면 F가 **합성곱 한 번 + 배치 정규화**로 너무 단순해져, 블록이 거의 항등 함수에 가까워지고 표현력이 떨어진다. 원 논문이 합성곱 **두 개**를 묶는 이유가 여기에 있다.

## 연습 8-14

[도전 문제] MiniResNet은 잔차 블록 세 개로 이뤄진 비교적 얕은 모델이다. 풀링 계층 사이에 입력과 출력 채널 수가 같은 잔차 블록(예: ResidualBlock(32, 32))을 여러 개 더 쌓아 훨씬 깊은 모델을 만들고, [연습 문제 8-2], [연습 문제 8-6]에서처럼 학습시킨 뒤 다음 질문에 답해 보자.

계층을 깊게 쌓아도 학습이 안정적으로 이뤄지는지, 그리고 분류 성능은 어떻게 달라지는지 MiniResNet과 비교해 확인해 보자.

확인한 결과를 바탕으로 모델의 성능을 높이기 위한 방법을 제시해 보자.

In [ ]:
def make_deep_resnet(n_per_stage=3):
    layers, fan_in = [nn.Conv2d(3, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU()], 32
    for ch in (32, 64, 128):
        layers.append(ResidualBlock(fan_in, ch))
        for _ in range(n_per_stage - 1):
            layers.append(ResidualBlock(ch, ch))      # 채널 수가 같은 블록을 더 쌓는다
        layers.append(nn.MaxPool2d(2)); fan_in = ch
    layers += [nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(fan_in, 10)]
    return nn.Sequential(*layers)

for n_blocks in (1, 3, 5):
    torch.manual_seed(SEED)
    model = make_deep_resnet(n_blocks)
    n_conv = sum(1 for m in model.modules() if isinstance(m, nn.Conv2d))
    print(f'[스테이지당 블록 {n_blocks}개 / 합성곱 {n_conv}개 / '
          f'파라미터 {sum(p.numel() for p in model.parameters()):,}개]')
    fit(model, epochs=10)

실행 결과는 다음과 같다.

| 스테이지당 블록 수 | 합성곱 계층 | 파라미터 | 평가 정확도 |
|---|---|---|---|
| 1개 | 9개 | 309,322 | 79.09% |
| 3개 | 21개 | 1,086,154 | 81.52% |
| 5개 | 33개 | 1,862,986 | **82.76%** |

잔차 연결 덕분에 **깊이를 늘릴수록 정확도가 계속 올라간다**. 합성곱 계층을 9개에서 33개까지 늘렸는데도 성능 저하가 전혀 나타나지 않는다. [연습 문제 8-2]에서 VGG 방식으로 깊이를 늘렸을 때 성능이 떨어졌던 것과 정반대다.

다만 향상 폭은 점점 줄어든다(79.09 → 81.52 → 82.76). 파라미터는 6배로 늘었는데 정확도는 3.7%p 오르는 데 그쳤다. 깊이의 이점을 계속 살리려면 **데이터의 양과 다양성도 함께 커져야 한다**.

## 연습 8-15

[도전 문제] [연습 문제 8-8]에서 만든 오토인코더의 인코더와 디코더를 모두 ResNet처럼 지름길을 가지도록 수정한 후 결과를 확인해 보자.

In [ ]:
class ResidualAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(),
            ResidualBlock(32, 32), nn.MaxPool2d(2),      # 32 -> 16
            ResidualBlock(32, 64), nn.MaxPool2d(2))      # 16 -> 8
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, 2, 1, output_padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), ResidualBlock(32, 32),
            nn.ConvTranspose2d(32, 16, 3, 2, 1, output_padding=1),
            nn.BatchNorm2d(16), nn.ReLU(), ResidualBlock(16, 16),
            nn.Conv2d(16, 3, 3, 1, 1), nn.Sigmoid())
    def forward(self, x): return self.decoder(self.encoder(x))

tf = transforms.ToTensor()
loader = DataLoader(datasets.CIFAR10(root=DATA_ROOT, train=True, download=True,
                                     transform=tf), batch_size=128, shuffle=True)
torch.manual_seed(SEED)
ae = ResidualAE().to(device)
crit = nn.MSELoss(); opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
for e in range(1, 6):
    ae.train(); tot = n = 0
    for x, _ in loader:
        x = x.to(device)
        loss = crit(ae(x), x)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(x); n += len(x)
    print(f'{e}/5 복원 손실 {tot / n:.5f}')

오토인코더에 지름길을 넣으면 복원 손실이 더 빠르게 떨어진다. 인코더와 디코더를 합쳐 계층이 깊어지는 구조라 기울기 전달이 어려운데, 잔차 연결이 이를 완화하기 때문이다.

실제로 U-Net처럼 인코더와 디코더 **사이를 직접 잇는** 지름길(스킵 연결)을 두면 세부 정보까지 보존되어 복원 품질이 크게 올라간다.

## 연습 8-16

[도전 문제] 각주 17에서 언급한 병목 구조는 ResNet-50, ResNet-101, ResNet-152 같은 깊은 ResNet이 연산량을 줄이려고 사용하는 잔차 블록의 변형이다. 다음 세 단계로 구성된 잔차를 만든다.

1x1 합성곱: 채널 수를 압축한다(출력 채널의 1/4이 표준).

3x3 합성곱: 압축된 채널에서 특징을 추출한다.

1x1 합성곱: 채널 수를 원래 출력 채널 수로 복원한다.

이 구조를 가진 잔차 블록 BottleneckBlock 클래스를 정의하자. 그런 다음 MiniResNet의 ResidualBlock을 BottleneckBlock으로 교체한 새 모델을 만들어, 다음 관점에서 MiniResNet과 비교해 보자.

전체 파라미터 수와 메모리 사용량

모델의 성능

In [ ]:
class BottleneckBlock(nn.Module):
    """1x1(압축) -> 3x3(연산) -> 1x1(복원) 구조의 잔차 블록"""
    def __init__(self, fan_in, fan_out, stride=1):
        super().__init__()
        mid = fan_out // 4                       # 출력 채널의 1/4로 압축
        self.body = nn.Sequential(
            nn.Conv2d(fan_in, mid, 1), nn.BatchNorm2d(mid), nn.ReLU(),
            nn.Conv2d(mid, mid, 3, stride, 1), nn.BatchNorm2d(mid), nn.ReLU(),
            nn.Conv2d(mid, fan_out, 1), nn.BatchNorm2d(fan_out))
        self.relu = nn.ReLU()
        self.shortcut = nn.Sequential()
        if stride != 1 or fan_in != fan_out:
            self.shortcut = nn.Sequential(nn.Conv2d(fan_in, fan_out, 1, stride),
                                          nn.BatchNorm2d(fan_out))
    def forward(self, x): return self.relu(self.body(x) + self.shortcut(x))

for name, blk in [('기본 블록', ResidualBlock), ('병목 블록', BottleneckBlock)]:
    b = blk(128, 128)
    print(f'{name}: 파라미터 {sum(p.numel() for p in b.parameters()):,}개')

for name, blk in [('기본 블록', ResidualBlock), ('병목 블록', BottleneckBlock)]:
    torch.manual_seed(SEED)
    print(f'[{name}]')
    fit(make_resnet(blk), epochs=10)

병목 구조는 3×3 합성곱을 **채널이 적은 상태에서만** 수행해 연산량을 크게 줄인다. 같은 채널 수에서 기본 블록보다 파라미터가 훨씬 적으면서 표현력은 유지되므로, ResNet-50 이상의 깊은 모델은 모두 이 구조를 쓴다.